# 2-motor: inductance

Measure the winding's inductance and the time constant of its current rise from a duty step into the locked rotor, with the ADC free running on the shunt and both terminals, and look under the one-pole rise for a slower component.

## Purpose

The current in a winding does not follow the voltage at once. It rises with a time constant $\tau = L / R$, the inductance over the resistance, and that time constant sets how much the current ripples inside one PWM window, how fast a duty step turns into torque, and how fast a current loop can be closed. The stage 5 current loop and the stage 6 trip timing are designed on it, and the in situ identification of stage 3 has to find the same number through a gear train and a case, so it is measured first on the bare motor where nothing else is in the way. A brushed motor with an iron rotor is not one pole either. Eddy currents in the laminations and the brush contact can add slower terms, and a design that assumes one pole where there are two reads the inductance from whatever stretch of the rise it happened to fit. So the second question here is whether a slow component sits under the pole, and how large it is.

The rotor is locked by the rig's bar, so there is no back-EMF (electromotive force, the voltage a turning rotor generates) and the terminal voltage divides only between the resistance and the inductance. Four holes of the bar, a quarter turn in 30 degree steps, put four commutator seats under the brushes, since the winding the terminals see changes with the rotor's angle. Two duties and both signs separate what scales with the step from what depends on the seat.

What earlier work established, in conclusions only. A duty ramp read through the control-rate samples gave no inductance. The fitted time constant grew with the length of the fit and never flattened, and after a step the current kept creeping for 100 to 200 ms, far beyond any one-pole rise, with eddy currents in the rotor iron and a contact settling under its new current as the candidates. A free-running burst into the locked rotor shows the rise at a resolution the control-rate samples never have, and a burst held long enough would show the slow part directly. This experiment is that burst at the length the capture allows.

## Procedure

The bare motor in the rig with its leads on the board's motor connector, the bar on the pinion at hole 0 with its clamp snug, the board on the DPS with the adapter on its bus at 3 Mbps, and no meter. `capture.py` powers the board and trims the DPS until the rail reads 5.0 V, then at each hole runs `osc-lab burst` with the shunt and both taps interleaved, at 25 and 40 % duty, both signs, five captures each, and rests the motor for twenty seconds before the bar is moved.

1. The motor in the rig, leads on the connector, the bar at hole 0 with its clamp snug.
2. `capture.py`. It asks for the leads on the DPS, powers up, and prompts at each of holes 1, 2 and 3 for the pins to come out, the bar to be turned to the hole and the pins to go back in, nothing pushing along the shaft.

Each burst is about a millisecond of drive at under half an amp, and the twenty bursts of a hole come at least 100 ms apart, so the copper stays within a fraction of a degree of the can.

## Data

- `hole-<K>/burst-<k>.csv`: one file per burst, twenty per hole. `k` is the sample index, `code` the raw ADC sample, and the meta fields repeat on the first row. `step_q15` is the step's duty and sign, `step_index` the sample at which the step was applied, `start_cnt` and `start_dir` the PWM counter at the first sample, `pwm_arr` the counter's crest, `chans` and `frame_len` the frame, `vbus_raw` the rail read before the arm, `bias` and `vmotor_bias` the shunt and tap bias readings, `pos` the pot and `drive_polarity` the wiring's sign.
- `hole-<K>/meta.json`: the servo's sense constants and the run's settings, not read here.

The capture had not been run when this notebook was written, so the assumptions about the burst's timing are stated here and are checked against the first two figures once it has.

- A frame is the shunt, tap A, the shunt, tap B, one conversion every 26 ADC clocks at 24 MHz, which is 52 timer ticks or 1.083 us at 48 ticks per microsecond. The shunt is every second sample and each tap every fourth, so the shunt is read 23 times per 50 us PWM period and each tap 11 to 12 times.
- A burst holds 960 samples, 1.04 ms, and the step is applied at the buffer's half, so the first half is the amplifier's zero and the second half, about 0.52 ms or ten PWM periods, is the rise.
- The duty written at the step takes effect at the next PWM update, at the crest or the trough, so the first driven window may be a half window. The analysis starts its period grid at the first commanded window start after the first trough after the step sample, and lets the fit's time origin float by up to a period to absorb whatever the first window did.
- The phase of each sample comes from the counter at the first sample and the fixed sample spacing, as `oscnb.bursts.phase` computes it, with the aperture of 13.5 ADC clocks added so a sample is placed where its aperture closed.
- The decay is slow, so the drive window is centered on the crest and between windows the bridge shorts the winding through its low side. The shunt then carries the winding current only inside the window, and the terminals read zero between windows. The fold of one burst below shows whether that holds.

### One burst

The first forward 25 % burst at hole 0 as recorded, the shunt and the two taps against time from the step.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.optimize import least_squares

from oscnb import boards, bursts as bu, motors, results
from oscnb.display import show

BLUE, ORANGE, INK, MUTED = "#2a78d6", "#eb6834", "#0b0b0b", "#898781"   # the house colours
MOTOR = motors.MOTORS[motors.DEFAULT]                                  # the motor, a parts count and nothing measured
BOARD = boards.BOARDS[boards.DEFAULT]                                  # the board, with the parts' nominal values
CAPTURE = (Path("../../../telemetry") / f"{MOTOR.key}__{BOARD.key}" / "inductance" / "capture-1").resolve()
HOLES = [0, 1, 2, 3]                                                   # the bar's holes, a quarter turn in 30 degree steps
ARR = BOARD.pwm_arr                                                    # ticks from the trough to the crest
PERIOD = 2 * ARR                                                       # ticks per PWM period, trough to trough
US = bu.STEP_TICKS / bu.TICKS_PER_US                                   # microseconds from one sample to the next
PERIOD_US = PERIOD / bu.TICKS_PER_US                                   # microseconds per PWM period
print(MOTOR.key, "on", BOARD.key, "from", results.rel(CAPTURE))
print(f"a sample every {US:.3f} us, {PERIOD / bu.STEP_TICKS:.1f} samples per {PERIOD_US:.0f} us period, {bu.CONVERSION_CLK} ADC clocks each")

# the stage 1 conversions this notebook reads counts through: measured, from the result files, never the registry's nominal
CS = results.read("1-board", "current-scale", pair=BOARD.key)         # counts per amp on the meter
TC = results.read("1-board", "terminal-chain", pair=BOARD.key)        # terminal counts per volt on the meter
ST = results.read("1-board", "sample-timing", pair=BOARD.key)         # the bridge's edge lags and the shunt's settle, per rail
COUNTS_PER_A = CS["counts_per_a"]["value"]
V_PER_COUNT = 1 / TC["terminal_counts_per_v"]["value"]
# 5.0 V sits between the two rails sample-timing graded, so the lags are the mean of the two; the period mean does not depend on them
ON_LAG = np.mean([ST["on_edge_lag_7v4"]["value"], ST["on_edge_lag_3v7"]["value"]])            # the shunt's half crossing, amplifier included
TAP_ON_LAG = np.mean([ST["tap_on_edge_lag_7v4"]["value"], ST["tap_on_edge_lag_3v7"]["value"]])   # the bridge's own edge, from the tap
OFF_LAG = np.mean([ST["off_edge_lag_7v4"]["value"], ST["off_edge_lag_3v7"]["value"]])
SETTLE = np.mean([ST["shunt_settle_7v4"]["value"], ST["shunt_settle_3v7"]["value"]])
print(f"current {COUNTS_PER_A:.1f} counts/A ({CS['counts_per_a']['lo']:.1f} to {CS['counts_per_a']['hi']:.1f}), "
      f"terminals {1 / V_PER_COUNT:.1f} counts/V, bridge ON edge {TAP_ON_LAG:.0f} ticks after the commanded start, shunt crossing {ON_LAG:.0f}, "
      f"OFF edge {OFF_LAG:.0f} after the commanded end, shunt settle {SETTLE:.0f}")

In [ ]:
rows = []
for h in HOLES:
    for k, df in enumerate(bu.load(CAPTURE / f"hole-{h}")):
        m = bu.meta(df)
        rows.append({"hole": h, "k": k, "pct": int(round(abs(m["step_q15"]) * 100 / 32768)),
                     "sign": 1 if m["step_q15"] > 0 else -1, "samples": len(df), "step_index": m["step_index"],
                     "frame_len": m["frame_len"], "rail_v": float(BOARD.rail_v(m["vbus_raw"])), "df": df})
B = pd.DataFrame(rows)
DUTIES = sorted(B["pct"].unique())
COLOR = dict(zip(DUTIES, (BLUE, ORANGE)))                              # one colour per duty, fixed through the notebook
summary = (B.groupby(["hole", "pct", "sign"]).agg(captures=("k", "size"), samples=("samples", "first"), frame_len=("frame_len", "first"),
                                                 step_index=("step_index", "mean"), rail_v=("rail_v", "mean")).reset_index())
show(summary, n=None)
print(f"{len(B)} bursts, rail {B['rail_v'].min():.2f} to {B['rail_v'].max():.2f} V before the arm, step at sample {B['step_index'].min()} to {B['step_index'].max()} of {B['samples'].iloc[0]}")

def one(hole, pct, sign, k=0):
    # the k-th capture of one hole, duty and sign
    return B[(B["hole"] == hole) & (B["pct"] == pct) & (B["sign"] == sign)].iloc[k]

ex = one(0, DUTIES[0], 1)
df = ex["df"]
m, ph, sh, ta, tb = bu.phase(df, ARR)
t = (np.arange(len(df)) - m["step_index"]) * US                        # microseconds from the step sample
y = df["code"].to_numpy(float)
fig, ax = plt.subplots(2, 1, figsize=(13, 6.5), sharex=True)
ax[0].plot(t[sh], y[sh], ".", ms=2.5, color=COLOR[ex["pct"]])
ax[0].axvline(0, color=INK, lw=0.8); ax[0].set_ylabel("shunt (counts)")
ax[0].set_title(f"hole {ex['hole']}, {ex['pct']} % forward, capture {ex['k']}: the shunt through the burst", loc="left", fontsize=10)
ax[1].plot(t[ta], y[ta], ".", ms=2.5, color=BLUE, label="tap A"); ax[1].plot(t[tb], y[tb], ".", ms=2.5, color=ORANGE, label="tap B")
ax[1].axvline(0, color=INK, lw=0.8); ax[1].set_ylabel("taps (counts)"); ax[1].legend(fontsize=8, loc="upper left")
ax[1].set_xlabel("time from the step sample (us)")
for a in ax:
    a.grid(alpha=0.3)
plt.tight_layout(); plt.show()
print(f"zero {bu.zero(df, ARR):.1f} counts over the pre-step half; {(len(df) - m['step_index']) * US:.0f} us after the step, "
      f"{(len(df) - m['step_index']) * US / PERIOD_US:.1f} periods")

The amplifier's zero for half a millisecond, then a window every 50 us with the current ramping inside it and the ripple's peak climbing from window to window. Tap A is the driven terminal and sits near the rail inside each window, tap B on the low side, and both return to the bias node between windows.

### The same burst folded onto the PWM period

Every sample after the step at its phase in the period, colored from the first driven period to the last, the shunt on the left and the terminal difference on the right.

In [ ]:
def samples(df):
    # one row per sample: its stream, its code, the tick its aperture closed on an unwrapped axis, its phase from the
    # crest, and its period on a grid that starts at the first commanded window start after the first trough after the step
    m, ph, sh, ta, tb = bu.phase(df, ARR)
    half = int(BOARD.drive_ticks(m["step_q15"]))                         # the window's half-width in ticks
    k = np.arange(len(df))
    phi0 = m["start_cnt"] if m["start_dir"] == 0 else PERIOD - m["start_cnt"]   # the counter's phase at the first sample, 0 at the trough
    tick = phi0 + bu.STEP_TICKS * k + bu.APERTURE_CLK * bu.TICKS_PER_ADC_CLK   # when each aperture closed, never wrapped
    trough = PERIOD * np.ceil(tick[m["step_index"]] / PERIOD)             # the first trough after the step sample
    origin = trough + ARR - half                                          # the commanded start of the window after it
    n = np.floor((tick - origin) / PERIOD).astype(int)                    # period index on the window-aligned grid, negative before it
    last = int(np.floor((tick[-1] - origin) / PERIOD))                    # the partial period at the end is dropped
    tc = ((tick - ARR) % PERIOD + PERIOD // 2) % PERIOD - PERIOD // 2     # ticks from the nearest crest
    stream = np.where(sh, "sh", np.where(ta, "ta", "tb"))
    s = pd.DataFrame({"k": k, "stream": stream, "code": df["code"].to_numpy(float), "tick": tick, "tc": tc, "n": n,
                      "t_us": (tick - origin) / bu.TICKS_PER_US, "full": n < last})
    return s, m, half

s, m, half = samples(df)
zero = bu.zero(df, ARR)
post = s[(s["n"] >= 0) & s["full"]]
N = post["n"].max() + 1
shade = plt.cm.Blues(np.linspace(0.3, 1.0, N)) if COLOR[ex["pct"]] == BLUE else plt.cm.Oranges(np.linspace(0.3, 1.0, N))
fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
for n, g in post.groupby("n"):
    g_sh = g[g["stream"] == "sh"]
    ax[0].plot(g_sh["tc"], g_sh["code"] - zero, ".", ms=4, color=shade[n], label=f"period {n}" if n in (0, N - 1) else None)
    # the taps are read 52 ticks apart, so the difference is drawn at tap A's phase against the nearest tap B sample
    a, b = g[g["stream"] == "ta"], g[g["stream"] == "tb"]
    vb = np.interp(a["tick"], b["tick"], b["code"]) if len(b) > 1 else np.nan
    ax[1].plot(a["tc"], ex["sign"] * (a["code"] - vb) * V_PER_COUNT, ".", ms=4, color=shade[n])
for a in ax:
    a.axvspan(-half, half, color=MUTED, alpha=0.15)                      # the commanded window
    a.axvline(-half + ON_LAG, color=INK, lw=0.6, ls=":"); a.axvline(half + OFF_LAG, color=INK, lw=0.6, ls=":")   # the edges sample-timing found
    a.axhline(0, color=INK, lw=0.5); a.set_xlabel("ticks from the crest (48 ticks = 1 us)"); a.grid(alpha=0.3); a.set_xlim(-ARR, ARR)
ax[0].set_ylabel("shunt over the zero (counts)"); ax[0].legend(fontsize=8, loc="upper left")
ax[0].set_title(f"the shunt, period by period: {N} full periods after the step", loc="left", fontsize=10)
ax[1].set_ylabel("terminal difference (V)"); ax[1].set_title("the terminal difference, the same periods", loc="left", fontsize=10)
plt.tight_layout(); plt.show()
off = post[(post["stream"] == "sh") & ((post["tc"] > half + OFF_LAG + 100) | (post["tc"] < -half - 100))]
print(f"the shunt between windows, well clear of the edges: {off['code'].mean() - zero:+.1f} counts over the zero (sd {off['code'].std():.1f}), "
      f"against the rest floor of about a count")

Each period draws the window once, the commanded window shaded and the dotted lines the edges sample-timing found on a resistor, and the color walks from light to dark as the current rises. The thing to see on the left is what the shunt reads between windows. A flat zero means the whole winding current crosses the shunt inside the window. A level over the zero after the OFF edge is charge the shunt's ground island hands back, which the ON window alone would miss and the period mean counts. On the right the terminal difference is the rail inside the window and near zero outside it, where the bridge shorts the winding.

## Method

The winding is a resistance and an inductance in series, with the rotor locked so no speed voltage enters:

$$v = R\,i + L\,\frac{di}{dt}$$

Under a fixed PWM duty the applied voltage repeats every period, and the response of this one-pole circuit to a voltage that starts at $t_0$ is its periodic steady state less a term that decays with the one time constant,

$$i(t) = i_{ss}(t) - A\,e^{-(t - t_0)/\tau}, \qquad \tau = \frac{L}{R}$$

whatever the switching inside the period does. The mean of $i$ over one period then rises as

$$\bar{i}_n = \frac{V}{R}\left(1 - e^{-(t_n - t_0)/\tau}\right)$$

with $V$ the mean applied voltage over the period and $t_n$ the period's center, which is the form the fit uses. $V / R$ is the settled current, and the fit's asymptote with the terminal difference the same burst reads gives $R$. The one-pole time constant $\tau$ is the parameter of interest, and $t_0$ floats by up to a period to absorb the first window being a half window and the ripple's shape. $L = R\,\tau$.

The period runs from the window's commanded start to the next, so each period holds one ON edge, one OFF edge and the whole brake interval after it. The mean over a whole period rather than the ON window's level is a charge balance. Whatever the shunt's ground island borrows during the ON edge it repays through the shunt before the period ends, and a linear amplifier's lag preserves area, so the period mean is the winding's current however the island and the amplifier behave inside the window. The fold figure above is where that is checked.

The period mean is not the mean of the period's samples. The sample spacing does not divide the period, 46.15 samples per 50 us, so counting samples reads a 25 % window as 11 or 12 samples long when it is 11.5, half a sample of alias, about 2 % of the level, and the alias drifts through the burst as a slow sawtooth that would read as a slow component. So each period is read in two parts weighted by their durations. The window between the bridge's edges, placed from the commanded edges by the tap's ON lag and the OFF lag of the sample-timing result, carries the current of a line through the window's settled samples, since the ramp over 25 us is close to straight. The OFF stretch carries the mean of the samples clear of both edges, which is the island's repayment if there is one. The shunt's counts over the burst's own zero go through the counts per amp of the current-scale result, and the tap difference, read the same way in two parts, through the terminal-chain result.

Three things come out of each burst. The fit over every full period gives $\tau$, $R$ and $L$. The same fit over the first 4, 5 and more periods gives $\tau$ against the fit length, and a one-pole winding gives the same $\tau$ at every length, the flat band the duty ramp never had. And the residual of the one-pole fit over the last three periods, as a share of the settled current, is the slow component's share at the burst's length. A one-pole winding leaves it at the noise, a slow term leaves the current still climbing over the fit. With ten periods of drive against an expected time constant of a few periods, the burst ends near three time constants rather than five, so the share is what the data holds and not the residual after five time constants the design asks for, which the limitations return to.

Five captures per hole, duty and sign give five values of each quantity. The error bar is the agreement between those captures, reported as the median with the range, and not a standard deviation, which five values do not support. The same per hole, per duty and per sign, and the spread across the holes is what the commutator seat does to the number.

### The period means of one burst

The grid and the fit on the burst above, and on the first forward burst at the higher duty beside it.

In [ ]:
def periods(df, sign):
    # one row per full period after the step: the winding current and the applied volts as period means, the period's center time.
    # The sample spacing does not divide the period, 46.15 samples per period, so a mean over the period's samples counts the
    # window 11 or 12 samples long when it is 11.5: half a sample of alias, 2 % of the level, drifting through the burst in a slow
    # sawtooth that would read as a slow component. So the window is read between the bridge's edges from a line through its
    # settled samples, and the OFF stretch from the mean of the samples clear of both edges, each weighted by its own duration.
    s, m, half = samples(df)
    zero = bu.zero(df, ARR)                                                # the amplifier's zero from the pre-step shunt samples
    a, b = -half + TAP_ON_LAG, half + OFF_LAG                              # the bridge's ON and OFF edges in ticks from the crest
    on_span = b - a
    s = s[(s["n"] >= 0) & s["full"]].assign(y=lambda d: d["code"] - zero)
    settled = (s["tc"] >= -half + SETTLE) & (s["tc"] <= half)              # inside the window, after the amplifier's settle, before the OFF edge
    clear = (s["tc"] >= b + SETTLE - ON_LAG) | (s["tc"] <= a - 10)        # the OFF stretch, clear of the OFF edge's tail and the next ON edge
    rows = []
    for n, g in s.groupby("n"):
        sh = g[(g["stream"] == "sh") & settled[g.index]]
        line = np.polyfit(sh["tc"], sh["y"], 1)                            # the ramp inside the window, which a line follows over 25 us
        i_on = np.polyval(line, (a + b) / 2)                               # its mean over the ON span, in counts over the zero
        i_off = g[(g["stream"] == "sh") & clear[g.index]]["y"].mean()      # what the shunt carries between windows
        ta_on, tb_on = (g[(g["stream"] == x) & settled[g.index]]["code"].mean() for x in ("ta", "tb"))
        ta_off, tb_off = (g[(g["stream"] == x) & clear[g.index]]["code"].mean() for x in ("ta", "tb"))
        v_on, v_off = sign * (ta_on - tb_on) * V_PER_COUNT, sign * (ta_off - tb_off) * V_PER_COUNT
        rows.append({"n": n, "t_us": (n + 0.5) * PERIOD_US,
                     "i_a": (i_on * on_span + i_off * (PERIOD - on_span)) / PERIOD / COUNTS_PER_A,
                     "v_v": (v_on * on_span + v_off * (PERIOD - on_span)) / PERIOD,
                     "i_on_a": i_on / COUNTS_PER_A, "i_off_a": i_off / COUNTS_PER_A, "v_on_v": v_on, "v_off_v": v_off,
                     "ramp_a_per_us": line[0] * bu.TICKS_PER_US / COUNTS_PER_A, "on_n": len(sh)})
    return pd.DataFrame(rows), zero, half

def rise(t, p):
    # the one-pole rise of the period mean: I_inf, tau in us, t0 in us
    return p[0] * (1 - np.exp(-(t - p[2]) / p[1]))

def fit_rise(P, m_periods=None):
    # least squares of the rise through the first m_periods period means (all of them by default); I_inf, tau and t0 free
    g = P if m_periods is None else P[P["n"] < m_periods]
    t, i = g["t_us"].to_numpy(), g["i_a"].to_numpy()
    start = [max(i[-1], 1e-3) * 1.2, 3 * PERIOD_US, 0.0]
    f = least_squares(lambda p: rise(t, p) - i, start, bounds=([0, 5, -PERIOD_US], [np.inf, 20 * PERIOD_US, PERIOD_US]))
    r = i - rise(t, f.x)
    return {"i_inf": f.x[0], "tau_us": f.x[1], "t0_us": f.x[2], "rms_a": float(np.sqrt(np.mean(r ** 2))), "resid": r, "periods": len(g)}

fig, ax = plt.subplots(2, 2, figsize=(14, 7), sharex=True, gridspec_kw={"height_ratios": [2, 1]})
for col, pct in enumerate(DUTIES):
    e = one(0, pct, 1)
    P, zero, half = periods(e["df"], e["sign"])
    f = fit_rise(P)
    v = P["v_v"].mean(); r_ohm = v / f["i_inf"]
    tt = np.linspace(0, P["t_us"].max() + PERIOD_US, 200)
    ax[0, col].plot(P["t_us"], P["i_a"] * 1e3, "o", color=COLOR[pct], label="period means")
    ax[0, col].plot(tt, rise(tt, [f["i_inf"], f["tau_us"], f["t0_us"]]) * 1e3, "-", color=INK, lw=1,
                    label=f"one pole: tau {f['tau_us']:.0f} us, settles at {f['i_inf'] * 1e3:.0f} mA, t0 {f['t0_us']:+.0f} us")
    ax[0, col].axhline(f["i_inf"] * 1e3, color=MUTED, lw=0.8, ls="--")
    ax[0, col].set_ylabel("period mean current (mA)"); ax[0, col].legend(fontsize=8, loc="lower right"); ax[0, col].grid(alpha=0.3)
    ax[0, col].set_title(f"hole 0, {pct} % forward: V {v:.2f} V, R {r_ohm:.2f} ohm, L {r_ohm * f['tau_us'] * 1e-3:.2f} mH", loc="left", fontsize=10)
    ax[1, col].plot(P["t_us"], f["resid"] * 1e3, "o-", color=COLOR[pct], lw=0.8)
    ax[1, col].axhline(0, color=INK, lw=0.5); ax[1, col].set_ylabel("residual (mA)"); ax[1, col].set_xlabel("time from the first window start (us)"); ax[1, col].grid(alpha=0.3)
    print(f"{pct} %: {f['periods']} periods, {P['on_n'].min()} to {P['on_n'].max()} settled shunt samples in a window, applied {v:.3f} V against "
          f"{pct / 100 * e['rail_v']:.3f} V of duty x rail, between windows {P['v_off_v'].mean():.3f} V and {P['i_off_a'].mean() * 1e3:+.1f} mA on the shunt, "
          f"rms {f['rms_a'] * 1e3:.1f} mA, last three periods {np.mean(f['resid'][-3:]) / f['i_inf'] * 100:+.1f} % of the asymptote")
plt.tight_layout(); plt.show()

The period means climb toward the asymptote with the one-pole curve through them, 25 % on the left and 40 % on the right, and the residuals under each. A residual that trends up over the late periods is the slow component. The printed line sets the applied volts the taps read against duty times the rail, the difference being the bridge's drops inside the window.

### Every burst

The fit over every burst of the dataset, one row per capture, then the inductance by hole.

In [ ]:
rows = []
for _, e in B.iterrows():
    P, zero, half = periods(e["df"], e["sign"])
    f = fit_rise(P)
    v = P["v_v"].mean(); r_ohm = v / f["i_inf"]
    rows.append({"hole": e["hole"], "pct": e["pct"], "sign": e["sign"], "k": e["k"], "periods": f["periods"], "zero": zero,
                 "v_mean_v": v, "i_inf_a": f["i_inf"], "tau_us": f["tau_us"], "t0_us": f["t0_us"], "rms_ma": f["rms_a"] * 1e3,
                 "r_ohm": r_ohm, "l_mh": r_ohm * f["tau_us"] * 1e-3, "slow_share": float(np.mean(f["resid"][-3:])) / f["i_inf"]})
FITS = pd.DataFrame(rows)
show(FITS, n=None)
G = FITS.groupby(["hole", "pct", "sign"]).agg(n=("k", "size"), tau_us=("tau_us", "median"), tau_lo=("tau_us", "min"), tau_hi=("tau_us", "max"),
                                             l_mh=("l_mh", "median"), l_lo=("l_mh", "min"), l_hi=("l_mh", "max"),
                                             r_ohm=("r_ohm", "median"), r_lo=("r_ohm", "min"), r_hi=("r_ohm", "max"), i_inf_a=("i_inf_a", "median"),
                                             slow_share=("slow_share", "median"), t0_us=("t0_us", "median"), periods=("periods", "median")).reset_index()

fig, ax = plt.subplots(1, 2, figsize=(14, 4.6))
MARK = {1: "o", -1: "^"}                                                  # forward filled circles, reverse open triangles
for pct in DUTIES:
    for sgn in (1, -1):
        g = FITS[(FITS["pct"] == pct) & (FITS["sign"] == sgn)]
        x = g["hole"] + (DUTIES.index(pct) - 0.5) * 0.3 + (sgn - 1) * 0.06          # a little room per duty and sign at each hole
        kw = dict(ls="none", marker=MARK[sgn], ms=5, color=COLOR[pct], mfc=COLOR[pct] if sgn == 1 else "none", alpha=0.8)
        ax[0].plot(x, g["l_mh"], label=f"{pct} % {'forward' if sgn == 1 else 'reverse'}", **kw)
        ax[1].plot(x, g["tau_us"], **kw)
    gg = G[G["pct"] == pct]
    xg = gg["hole"] + (DUTIES.index(pct) - 0.5) * 0.3 + (gg["sign"] - 1) * 0.06 + 0.12
    ax[0].errorbar(xg, gg["l_mh"], yerr=[gg["l_mh"] - gg["l_lo"], gg["l_hi"] - gg["l_mh"]], fmt="_", ms=10, color=INK, lw=1, capsize=3)
    ax[1].errorbar(xg, gg["tau_us"], yerr=[gg["tau_us"] - gg["tau_lo"], gg["tau_hi"] - gg["tau_us"]], fmt="_", ms=10, color=INK, lw=1, capsize=3)
ax[0].set_ylabel("L = R tau (mH)"); ax[0].set_title("the inductance per capture, by hole", loc="left", fontsize=10); ax[0].legend(fontsize=8)
ax[1].set_ylabel("tau (us)"); ax[1].set_title("the time constant per capture, by hole", loc="left", fontsize=10)
for a in ax:
    a.set_xticks(HOLES, [f"hole {h}" for h in HOLES]); a.grid(alpha=0.3, axis="y")
plt.tight_layout(); plt.show()
for pct in DUTIES:
    g = FITS[FITS["pct"] == pct]
    print(f"{pct} %: L {g['l_mh'].median():.3f} mH ({g['l_mh'].min():.3f} to {g['l_mh'].max():.3f}), tau {g['tau_us'].median():.0f} us "
          f"({g['tau_us'].min():.0f} to {g['tau_us'].max():.0f}), R {g['r_ohm'].median():.2f} ohm, t0 {g['t0_us'].median():+.0f} us, "
          f"slow share {g['slow_share'].median() * 100:+.1f} % ({g['slow_share'].min() * 100:+.1f} to {g['slow_share'].max() * 100:+.1f})")
print(f"by hole, L in mH: " + ", ".join(f"hole {h} {FITS[FITS['hole'] == h]['l_mh'].median():.3f}" for h in HOLES))

The inductance on the left and the time constant on the right, per capture against the hole, 25 % in blue and 40 % in orange, forward filled and reverse open, with each group's median and range as the black bar and whisker. Captures of one group agreeing within the whisker is the measurement. Groups moving with the hole is the commutator seat. Groups moving with the duty or the sign is what a one-pole winding does not do, and the printed lines give the numbers.

### The flat band

The same fit over the first 4, 5 and more periods of every burst, so $\tau$ can be read against the length of the fit.

In [ ]:
rows = []
for _, e in B.iterrows():
    P, zero, half = periods(e["df"], e["sign"])
    for mp in range(4, len(P) + 1):
        f = fit_rise(P, mp)
        rows.append({"hole": e["hole"], "pct": e["pct"], "sign": e["sign"], "k": e["k"], "fit_periods": mp, "tau_us": f["tau_us"], "i_inf_a": f["i_inf"]})
SCAN = pd.DataFrame(rows)
fig, ax = plt.subplots(1, 2, figsize=(14, 4.4))
for pct in DUTIES:
    g = SCAN[SCAN["pct"] == pct].groupby("fit_periods")
    for a, col, lab in ((ax[0], "tau_us", "tau (us)"), (ax[1], "i_inf_a", "fitted settled current (A)")):
        m_ = g[col].median(); lo = g[col].min(); hi = g[col].max()
        a.errorbar(m_.index + (DUTIES.index(pct) - 0.5) * 0.15, m_, yerr=[m_ - lo, hi - m_], fmt="o-", ms=4, lw=1, color=COLOR[pct], capsize=2, label=f"{pct} %")
        a.set_ylabel(lab)
for a in ax:
    a.set_xlabel("periods in the fit"); a.grid(alpha=0.3); a.legend(fontsize=8)
ax[0].set_title("tau against the fit length, every capture of a duty pooled", loc="left", fontsize=10)
ax[1].set_title("the asymptote against the fit length", loc="left", fontsize=10)
plt.tight_layout(); plt.show()
for pct in DUTIES:
    g = SCAN[SCAN["pct"] == pct].groupby("fit_periods")["tau_us"].median()
    print(f"{pct} %: tau {g.iloc[0]:.0f} us at {g.index[0]} periods, {g.iloc[-1]:.0f} us at {g.index[-1]}, {(g.iloc[-1] / g.iloc[0] - 1) * 100:+.0f} % over the scan")

The fitted $\tau$ on the left and the fitted asymptote on the right against the number of periods in the fit, pooled over the captures of each duty, the point the median and the whisker the range. A flat line is a one-pole winding. A line that keeps rising with the length is a slower component folding into $\tau$, the shape the duty ramp had, and the short fits are where the ripple's first periods weigh most.

### The resistance the burst reads

$R$ from the burst's asymptote per hole, beside the rest-resistance result for the same motor where it exists.

In [ ]:
RR_PATH = results.path("2-motor", "rest-resistance", pair=MOTOR.key)
RR = results.read("2-motor", "rest-resistance", pair=MOTOR.key) if RR_PATH.exists() else None
if RR is None:
    print("no rest-resistance result yet; the burst's R stands on its own")
else:
    print("rest-resistance carries:", {k: v.get("value") if isinstance(v, dict) else v for k, v in RR.items() if k not in ("capture", "notes")})
fig, ax = plt.subplots(figsize=(8, 4))
for pct in DUTIES:
    for sgn in (1, -1):
        g = G[(G["pct"] == pct) & (G["sign"] == sgn)]
        x = g["hole"] + (DUTIES.index(pct) - 0.5) * 0.2 + (sgn - 1) * 0.04
        ax.errorbar(x, g["r_ohm"], yerr=[g["r_ohm"] - g["r_lo"], g["r_hi"] - g["r_ohm"]], fmt=MARK[sgn], ms=5, color=COLOR[pct],
                    mfc=COLOR[pct] if sgn == 1 else "none", capsize=2, lw=1, label=f"{pct} % {'forward' if sgn == 1 else 'reverse'}")
ax.set_xticks(HOLES, [f"hole {h}" for h in HOLES]); ax.set_ylabel("R from the asymptote (ohm)"); ax.grid(alpha=0.3, axis="y"); ax.legend(fontsize=8)
ax.set_title("the resistance the burst reads, V over the settled current, by hole", loc="left", fontsize=10)
plt.tight_layout(); plt.show()
print("R by hole: " + ", ".join(f"hole {h} {FITS[FITS['hole'] == h]['r_ohm'].median():.2f} ohm ({FITS[FITS['hole'] == h]['r_ohm'].min():.2f} to {FITS[FITS['hole'] == h]['r_ohm'].max():.2f})" for h in HOLES))

The burst's $R$ per hole, duty and sign, each group's median with its range. It carries the brush drop at this current over the current, so it reads above a meter's ohms at a milliamp by that much, and the rest-resistance result for the same holes is the comparison once both exist.

## Results

The result file carries the inductance, the time constant, the burst's resistance and the slow share, each the median over every capture with the range of the group medians as its bounds, the settled current, and the same four quantities per hole as lists and per duty and sign as a table, so stage 3 and stage 5 can read the number and how far the seat moves it.

In [ ]:
def q(col, unit):
    # the median over every capture, bounded by the range of the group medians (hole x duty x sign)
    return results.quantity(FITS[col].median(), lo=G[col].min(), hi=G[col].max(), unit=unit)

def by_hole(col):
    return [float(FITS[FITS["hole"] == h][col].median()) for h in HOLES]

def by_duty_sign(col):
    return {f"{p}_{'fwd' if s > 0 else 'rev'}": float(v) for (p, s), v in FITS.groupby(["pct", "sign"])[col].median().items()}

results.write("2-motor", "inductance", CAPTURE, pair=MOTOR.key,
              notes="duty steps into the locked rotor at four holes, 25 and 40 % on a 5 V rail, both signs, five captures each; "
                    "one-pole fit of the period means, R from the asymptote and the tap difference, L = R tau; "
                    "bounds are the range of the group medians, the slow share is the residual over the last three periods",
              l_mh=q("l_mh", "mH"), tau_us=q("tau_us", "us"), r_burst_ohm=q("r_ohm", "ohm"), slow_share=q("slow_share", "fraction"),
              i_settled_a=q("i_inf_a", "A"), fit_origin_us=q("t0_us", "us"), periods_fitted=q("periods", "periods"),
              captures=results.quantity(len(FITS), unit="captures"),
              l_mh_by_hole=by_hole("l_mh"), tau_us_by_hole=by_hole("tau_us"), r_burst_ohm_by_hole=by_hole("r_ohm"), slow_share_by_hole=by_hole("slow_share"),
              l_mh_by_duty_sign=by_duty_sign("l_mh"), tau_us_by_duty_sign=by_duty_sign("tau_us"),
              r_burst_ohm_by_duty_sign=by_duty_sign("r_ohm"), slow_share_by_duty_sign=by_duty_sign("slow_share"))

## Limitations

- The burst holds about ten PWM periods of drive, near three time constants at the expected $\tau$. The one-pole fit is well set by that, the residual after five time constants is not in the data, and the slow share is defined over the last three periods of the fit instead. A burst with the step earlier in its buffer, or a longer buffer, would hold five.
- Four seats of twelve. The rotor's angle changes the winding the terminals see, and the burst reads the seat it is at. Four holes bound the spread without giving its shape.
- The applied voltage is the mean tap difference over the period, which holds the bridge's drops and not the brush drop, so $R$ from the burst is the winding plus the brushes at this current, and $L = R\,\tau$ carries that $R$.
- The two-part period mean still carries the samples' phase against the bridge's edges at the half-percent level, a slow wobble over the burst's periods that a resistor burst shows with no inductance in it. It sets the floor under the slow share, which the resistor bursts of sample-timing can be run through the same cells to read.
- The in-drive offset of about five counts the current-scale result found at 100 % duty is not subtracted. On a settled level of a few hundred counts it is under 2 % of $R$ and none of $\tau$.
- The rail is read before the arm with no load. A rail that sags under the burst lowers $V$ and the asymptote together, which the tap difference per period shows. A rail that sags over the rise itself would stretch $\tau$, which is in the open questions.
- 5.0 V sits between the two rails sample-timing graded, so the edge lags drawn on the fold are the mean of the two. The period mean does not depend on them.
- The DPS is the supply.

## Open questions

- Whether the slow component is in the rotor iron or the brush contact. Eddy currents scale with the step and are the same at every seat, a contact settling depends on the seat and on the current through it. The share per duty against the share per hole in this data is the first cut, and a burst held five time constants with the tap difference read per period would separate them.
- Whether the shunt reads a repayment between windows. The fold shows it. If it is there, the ON-window level under-reads the winding and the period mean is the route, and its size per duty says what the island holds.
- Whether $\tau$ depends on the step. A one-pole winding gives the same $\tau$ at 25 and 40 %, and an iron rotor's incremental inductance can fall as the current rises. The two duties here are the test, and a third duty up to the burst's applied-volts cap would extend it.
- Whether the rail holds through the rise. A burst with the rail tap in its frame would read it inside the window period by period, and the tap difference per period in this data is the first look.

## Conclusion

To be written from the capture.